<a href="https://colab.research.google.com/github/Hajer5503/Esprit-PI-4DS5-AgriSmart/blob/hajer-branch/modules/irrigation_rl/notebooks/agrismart_langgraph_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌾 AgriSmart — Autonomous Irrigation AI Agent
### LangGraph + RAG Knowledge Base + MLflow Tracking

## What's new vs previous version
| | Before | Now |
|---|---|---|
| Tools | 3 (forecast, RL, weather) | **4** (+ Agronomy RAG) |
| LLM output | Cut off at 20 tokens | **Fixed** (256 tokens, return_full_text=False) |
| Knowledge | LLM general internet | **FAO-56 grounded** via ChromaDB |
| Audit trail | None | **MLflow** logs every decision |
| Layer 2 model | sklearn DQN | **PyTorch Double DQN** (auto-detect) |

## Pipeline
```
Sensor data → Layer1 → RL Advisor → Weather → RAG → LLM → MLflow → Output
```

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain langchain-huggingface langchain-community transformers accelerate huggingface_hub chromadb sentence-transformers mlflow requests numpy pandas scikit-learn joblib torch
print('✅ All dependencies installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 511.1 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 106.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/

## 2. Imports

In [ ]:
import os, json, joblib, requests, re, math, copy, datetime
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from langchain_core.tools import tool
import mlflow
print("✅ Imports OK")

✅ Imports OK


## 3. HuggingFace Login & Load LLM

In [ ]:
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = API_KEY
from huggingface_hub import login
login(os.environ["HUGGINGFACEHUB_API_TOKEN"])

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

model_id  = "google/gemma-2-2b-it"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model     = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="auto")

# KEY FIX: max_new_tokens only, return_full_text=False
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=False,
    temperature=None,
    top_p=None,
    repetition_penalty=1.1,
    return_full_text=False,
)

class LLMWrapper:
    def __init__(self, p): self.pipe = p
    def invoke(self, prompt):
        result = self.pipe(prompt)
        text   = result[0]["generated_text"] if result else ""
        text   = re.sub(r"<[^>]+>", "", text).strip()
        class R:
            def __init__(self, t): self.content = t
        return R(text)

llm = LLMWrapper(pipe)
print("✅ LLM: Gemma-2-2b-it | max_new_tokens=256 | return_full_text=False")

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'top_p', 'do_sample', 'repetition_penalty', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ LLM: Gemma-2-2b-it | max_new_tokens=256 | return_full_text=False


## 4. Load ML Models

In [ ]:
l1_sm  = joblib.load('layer1_model_sm_root.pkl')
l1_et0 = joblib.load('layer1_model_et0.pkl')
with open('layer1_feature_cols.json')     as f: L1_FEATURES  = json.load(f)
with open('layer1_et0_feature_cols.json') as f: ET0_FEATURES = json.load(f)
print(f"Layer 1 sm_root : {type(l1_sm).__name__}  ({len(L1_FEATURES)} features)")
print(f"Layer 1 ET0     : {type(l1_et0).__name__}  ({len(ET0_FEATURES)} features)")

# Auto-detect PyTorch vs sklearn DQN
USE_PYTORCH = False
try:
    import torch
    import torch.nn as nn
    class QNetwork(nn.Module):
        def __init__(self, n_state=9, n_actions=5, hidden=128):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(n_state, hidden), nn.ReLU(),
                nn.Linear(hidden, 64), nn.ReLU(),
                nn.Linear(64, n_actions),
            )
        def forward(self, x): return self.net(x)

    ck = torch.load('layer2_dqn_agent_augmented.pt', map_location='cpu')
    dqn_net = QNetwork(9, 5)
    dqn_net.load_state_dict(ck['online'])
    dqn_net.eval()
    USE_PYTORCH = True
    print("Layer 2 DQN     : PyTorch Double DQN ✓")
except Exception as e:
    print(f"PyTorch not found ({e}), trying sklearn …")
    dqn_data = joblib.load('layer2_dqn_agent_augmented.pt')

    # ── Validate that nets are actually fitted ────────────────────────────────
    fitted_count = sum(dqn_data['fitted'])
    print(f"Layer 2 DQN     : sklearn DQN loaded  |  fitted nets: {fitted_count}/5")
    if fitted_count == 0:
        print("⚠  WARNING: sklearn DQN has 0 fitted nets → will always return 0mm")
        print("   Fix: upload layer2_dqn_agent_augmented.pt from Layer 2 PyTorch notebook")
    USE_PYTORCH = False

print("✅ All models ready")

Layer 1 sm_root : ExtraTreesRegressor  (74 features)
Layer 1 ET0     : HistGradientBoostingRegressor  (30 features)
Layer 2 DQN     : PyTorch Double DQN ✓
✅ All models ready


## 5. Constants

In [ ]:
FC, WP = 0.28, 0.12
TAW    = FC - WP
RAW    = 0.55 * TAW
SM_LOW = WP + RAW
SM_HIGH = FC
ACTIONS = [0, 5, 10, 15, 20]
N_ACTIONS = len(ACTIONS)
print(f"Optimal SM : [{SM_LOW:.3f}, {SM_HIGH:.3f}]  |  Actions: {ACTIONS} mm/day")

Optimal SM : [0.208, 0.280]  |  Actions: [0, 5, 10, 15, 20] mm/day


## 6. RAG — FAO-56 Knowledge Base

13 agronomic passages indexed in ChromaDB.
Retrieved automatically before each LLM explanation.

In [ ]:
import chromadb

FAO56_DOCS = [
    "The FAO-56 stress threshold for wheat is SM_LOW = WP + p*(FC-WP) where p=0.55. "
    "For sandy loam soil with FC=0.28 and WP=0.12, stress threshold = 0.208 vol/vol. "
    "Below 0.208 wheat experiences water stress and yield is reduced.",

    "Field Capacity (FC) for sandy loam soil is 0.28 vol/vol. Above FC water drains freely. "
    "The optimal soil moisture range for wheat in Tunisia is [0.208, 0.280] vol/vol.",

    "Wheat crop coefficient Kc by growth stage: Initial (days 1-20) Kc=0.70, "
    "Development (days 20-50) Kc rises to 1.15, Mid-season (days 50-175) Kc=1.15 peak demand, "
    "Late-season (days 175-228) Kc declines to 0.35 as crop matures.",

    "During wheat mid-season Kc=1.15 and ET0=5 mm/day gives ETc=5.75 mm/day maximum demand. "
    "This is the period of highest irrigation need in Tunisia (March-May).",

    "Irrigation should be applied before soil moisture drops below 0.208. "
    "For sandy loam in Tunisia, 10-20mm irrigation restores soil to optimal range.",

    "When rainfall exceeds 50% of ETc demand, irrigation can be skipped. "
    "Rainfall of 5mm or more in Tunisia winter typically covers daily ETc of 2-3mm.",

    "In Tunisia, wheat is sown in November and harvested in June. "
    "Winter months (Nov-Feb) ET0 = 1.8-2.6 mm/day with more rainfall. "
    "Spring months (Mar-May) require most irrigation, ET0 rises to 5-6 mm/day.",

    "Water stress during flowering (mid-season, days 50-100) reduces yield by 30-50%. "
    "Stress during maturity (days 175-228) has minimal yield impact.",

    "Monthly ET0 in Tunisia: Nov=2.6, Dec=1.8, Jan=2.1, Feb=2.4, "
    "Mar=3.9, Apr=4.6, May=5.3, Jun=6.2 mm/day.",

    "Vapor Pressure Deficit VPD is a key ET0 driver. "
    "VPD = (1 - RH/100) * 0.6108 * exp(17.27*T/(T+237.3)). "
    "High VPD above 2 kPa indicates high atmospheric demand.",

    "Effective rainfall for sandy loam: rainfall above 15mm has runoff. "
    "Light rainfall 1-3mm is absorbed but may be insufficient to cover ETc.",

    "The RL agent recommends one of 5 irrigation actions: 0, 5, 10, 15, or 20 mm/day. "
    "It was trained to keep soil moisture in [0.208, 0.280] across 1000+ simulated seasons.",

    "Total seasonal irrigation for wheat in Tunisia is typically 300-500mm depending on rainfall. "
    "Deficit irrigation targeting 80% field capacity can save 20-30% water with minimal yield loss.",
]

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("agrismart_fao56")
except: pass
collection = chroma_client.create_collection("agrismart_fao56", metadata={"hnsw:space":"cosine"})
collection.add(documents=FAO56_DOCS, ids=[f"doc_{i}" for i in range(len(FAO56_DOCS))])

def retrieve_knowledge(query, n=3):
    res = collection.query(query_texts=[query], n_results=n)
    return "\n".join([f"- {d}" for d in res['documents'][0]])

print(f"✅ RAG knowledge base ready  |  {len(FAO56_DOCS)} FAO-56 passages")
print(f"   Test: {retrieve_knowledge('wheat stress threshold')[:120]}...")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:08<00:00, 10.3MiB/s]


✅ RAG knowledge base ready  |  13 FAO-56 passages
   Test: - The FAO-56 stress threshold for wheat is SM_LOW = WP + p*(FC-WP) where p=0.55. For sandy loam soil with FC=0.28 and WP...


## 7. MLflow Setup

In [ ]:
mlflow.set_tracking_uri("file:./mlruns")
EXP_NAME = "AgriSmart_Irrigation_Agent"
try:
    mlflow.create_experiment(EXP_NAME)
except:
    pass
mlflow.set_experiment(EXP_NAME)
print(f"✅ MLflow ready  |  Experiment: {EXP_NAME}")
print(f"   Each agent call logs: params, metrics, tags, explanation text")

✅ MLflow ready  |  Experiment: AgriSmart_Irrigation_Agent
   Each agent call logs: params, metrics, tags, explanation text


## 8. Define the Four Tools

In [ ]:
@tool
def layer1_forecast(
    et0_mm: float, precip_mm: float, temp_c: float, rh_pct: float, wind_kmh: float,
    sm_root: float, sm_shallow: float, sm_deep: float, kc: float, etc_mm: float,
    irrigation_mm: float, day_of_season: int, month: int, date_str: str,
) -> dict:
    """Predicts tomorrow's sm_root and ET0 using Layer 1 ML models."""
    row = {
        'et0_mm':et0_mm,'precip_mm':precip_mm,'temp_c':temp_c,'rh_pct':rh_pct,
        'wind_kmh':wind_kmh,'kc':kc,'etc_mm':etc_mm,'sm_root':sm_root,
        'sm_shallow':sm_shallow,'sm_deep':sm_deep,'irrigation_mm':irrigation_mm,
        'day_of_season':day_of_season,'month':month,
    }
    for col in ['sm_root','sm_shallow','sm_deep','et0_mm','etc_mm','precip_mm','temp_c','irrigation_mm']:
        for lag in [1,3,7]: row[f'{col}_lag{lag}'] = row.get(col,0)
    for lag in [10,14]: row[f'et0_mm_lag{lag}'] = et0_mm
    for col,fs in [('sm_root',['mean','std']),('et0_mm',['mean','sum']),
                    ('precip_mm',['sum']),('temp_c',['mean']),('irrigation_mm',['sum'])]:
        for f in fs: row[f'{col}_roll7_{f}'] = row.get(col,0)
    doy = pd.Timestamp(date_str).dayofyear
    row['sin_doy'] = math.sin(2*math.pi*doy/365)
    row['cos_doy'] = math.cos(2*math.pi*doy/365)
    lat=math.radians(36.0); dr=1+0.033*math.cos(2*math.pi*doy/365)
    decl=0.409*math.sin(2*math.pi*doy/365-1.39)
    ws=math.acos(max(-1,min(1,-math.tan(lat)*math.tan(decl))))
    Ra=((24*60/math.pi)*0.0820*dr*(ws*math.sin(lat)*math.sin(decl)+math.cos(lat)*math.cos(decl)*math.sin(ws)))
    row['Ra']=Ra; row['et0_hargreaves']=(0.0023*(temp_c+17.8)*math.sqrt(max(temp_c*0.4,0.1))*Ra*0.408)
    row['Rs_MJ']=0.0; row['Rns']=0.0
    row['vpd']=(1-rh_pct/100)*0.6108*math.exp(17.27*temp_c/(temp_c+237.3))
    for lag in [1,3,7,10,14]:
        row[f'shortwave_radiation_lag{lag}']=0.0; row[f'Rs_MJ_lag{lag}']=0.0
    for lag in [1,3,7]:
        row[f'Ra_lag{lag}']=Ra; row[f'et0_hargreaves_lag{lag}']=row['et0_hargreaves']
        row[f'vpd_lag{lag}']=row['vpd']
    for lag in [10,14]: row[f'vpd_lag{lag}']=row['vpd']
    X_sm  = np.array([[row.get(f,0) for f in L1_FEATURES]])
    X_et0 = np.array([[row.get(f,0) for f in ET0_FEATURES]])
    pred_sm  = float(l1_sm.predict(X_sm)[0])
    pred_et0 = float(0.30*et0_mm + 0.70*l1_et0.predict(X_et0)[0])
    status = "STRESSED" if pred_sm < SM_LOW else ("SATURATED" if pred_sm > SM_HIGH else "OPTIMAL")
    gap    = round(abs(SM_LOW-pred_sm) if pred_sm < SM_LOW else (abs(pred_sm-SM_HIGH) if pred_sm > SM_HIGH else 0), 4)
    return {"pred_sm_root":round(pred_sm,4),"pred_et0_mm":round(pred_et0,4),
            "stress_level":status,"stress_gap":gap}

print("✅ Tool 1 (layer1_forecast) defined")

✅ Tool 1 (layer1_forecast) defined


In [ ]:
@tool
def rl_advisor(
    et0_mm: float, precip_mm: float, sm_shallow: float, sm_deep: float,
    kc: float, growth_stage: int, day_of_season: int,
    pred_sm_root: float, pred_et0_mm: float,
) -> dict:
    """Uses trained DQN to recommend optimal irrigation mm. Call layer1_forecast first."""
    state = np.array([et0_mm, precip_mm, sm_shallow, sm_deep, kc,
                      growth_stage/4.0, day_of_season/228.0,
                      pred_sm_root, pred_et0_mm/10.0], dtype=np.float32)
    etc_approx    = et0_mm * kc
    rain_override = (pred_sm_root >= SM_LOW and precip_mm >= etc_approx * 0.5)
    q_vals        = np.zeros(N_ACTIONS)
    if rain_override:
        action_idx = 0; irr_mm = 0
        constraint = "Rain constraint: rain covers ETc demand"
    elif USE_PYTORCH:
        with torch.no_grad():
            q_vals = dqn_net(torch.FloatTensor(state).unsqueeze(0)).squeeze().numpy()
        action_idx = int(np.argmax(q_vals)); irr_mm = ACTIONS[action_idx]; constraint = None
    else:
        sc = dqn_data['scaler'].transform(state.reshape(1,-1))
        for a in range(N_ACTIONS):
            if dqn_data['fitted'][a]: q_vals[a] = dqn_data['nets'][a].predict(sc)[0]
        action_idx = int(np.argmax(q_vals)); irr_mm = ACTIONS[action_idx]; constraint = None
    reason = (f"Soil STRESSED tomorrow ({pred_sm_root:.3f} < {SM_LOW}). Irrigate." if pred_sm_root < SM_LOW
              else f"Soil SATURATED ({pred_sm_root:.3f} > {SM_HIGH}). Skip." if pred_sm_root > SM_HIGH
              else f"Soil OPTIMAL ({pred_sm_root:.3f}). Maintenance.")
    return {"recommended_irrigation_mm":irr_mm,"reasoning":reason,
            "constraint_applied":constraint,
            "q_values":{f"{ACTIONS[i]}mm":round(float(q_vals[i]),3) for i in range(N_ACTIONS)}}

print("✅ Tool 2 (rl_advisor) defined")

✅ Tool 2 (rl_advisor) defined


In [ ]:
@tool
def weather_forecast(latitude: float, longitude: float, days_ahead: int = 1) -> dict:
    """Live weather forecast from Open-Meteo API (free). Default: Tunisia 36.8N, 10.2E"""
    url=(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}"
         f"&daily=precipitation_sum,temperature_2m_max,temperature_2m_min,"
         f"relative_humidity_2m_max,wind_speed_10m_max,et0_fao_evapotranspiration"
         f"&forecast_days={days_ahead+1}&timezone=Africa/Tunis")
    try:
        resp=requests.get(url,timeout=10); resp.raise_for_status()
        d=resp.json()['daily']; idx=min(days_ahead,len(d['time'])-1)
        p=d['precipitation_sum'][idx] or 0
        return {"forecast_date":d['time'][idx],"precipitation_mm":round(p,1),
                "temp_max_c":round(d['temperature_2m_max'][idx] or 0,1),
                "et0_forecast_mm":round(d['et0_fao_evapotranspiration'][idx] or 0,2),
                "rain_risk":"HIGH" if p>5 else "MEDIUM" if p>1 else "LOW"}
    except Exception as e:
        return {"error":str(e),"precipitation_mm":0.0,"rain_risk":"UNKNOWN"}

print("✅ Tool 3 (weather_forecast) defined")

✅ Tool 3 (weather_forecast) defined


In [ ]:
@tool
def agronomy_rag(question: str) -> dict:
    """Retrieves relevant FAO-56 agronomic knowledge from the knowledge base.
    Always call before generating the final explanation."""
    knowledge = retrieve_knowledge(question, n=3)
    return {"retrieved_passages":knowledge,
            "source":"FAO Irrigation and Drainage Paper No. 56",
            "n_passages":3}

print("✅ Tool 4 (agronomy_rag) defined")

✅ Tool 4 (agronomy_rag) defined


## 9. Main Agent Function

In [ ]:
def run_irrigation_agent(sensor_data: dict, farm_coords: dict = None) -> str:
    if farm_coords is None:
        farm_coords = {"lat": 36.8, "lon": 10.2}

    with mlflow.start_run(run_name="irrigation_" + sensor_data['date']) as run:

        mlflow.log_params({
            "date":sensor_data["date"],"sm_root":sensor_data["sm_root"],
            "et0_mm":sensor_data["et0_mm"],"precip_mm":sensor_data["precip_mm"],
            "kc":sensor_data["kc"],"growth_stage":sensor_data["growth_stage"],
            "day_of_season":sensor_data["day_of_season"],
        })

        print("  [1/4] Layer 1 forecast …")
        l1 = layer1_forecast.invoke({
            "et0_mm":sensor_data["et0_mm"],"precip_mm":sensor_data["precip_mm"],
            "temp_c":sensor_data["temp_c"],"rh_pct":sensor_data["rh_pct"],
            "wind_kmh":sensor_data["wind_kmh"],"sm_root":sensor_data["sm_root"],
            "sm_shallow":sensor_data["sm_shallow"],"sm_deep":sensor_data["sm_deep"],
            "kc":sensor_data["kc"],"etc_mm":sensor_data["etc_mm"],
            "irrigation_mm":sensor_data.get("irrigation_mm",0.0),
            "day_of_season":sensor_data["day_of_season"],
            "month":int(sensor_data["date"].split("-")[1]),
            "date_str":sensor_data["date"],
        })
        print(f"     pred_sm={l1['pred_sm_root']}  pred_et0={l1['pred_et0_mm']}  {l1['stress_level']}")
        mlflow.log_metrics({"pred_sm_root":l1["pred_sm_root"],"pred_et0_mm":l1["pred_et0_mm"]})
        mlflow.set_tag("stress_level", l1["stress_level"])

        print("  [2/4] RL advisor …")
        rl = rl_advisor.invoke({
            "et0_mm":sensor_data["et0_mm"],"precip_mm":sensor_data["precip_mm"],
            "sm_shallow":sensor_data["sm_shallow"],"sm_deep":sensor_data["sm_deep"],
            "kc":sensor_data["kc"],"growth_stage":sensor_data["growth_stage"],
            "day_of_season":sensor_data["day_of_season"],
            "pred_sm_root":l1["pred_sm_root"],"pred_et0_mm":l1["pred_et0_mm"],
        })
        print(f"     action={rl['recommended_irrigation_mm']}mm  constraint={rl['constraint_applied']}")
        mlflow.log_metric("irrigation_mm", rl["recommended_irrigation_mm"])
        mlflow.set_tag("rain_constraint", str(rl["constraint_applied"] is not None))

        print("  [3/4] Weather forecast …")
        wx = weather_forecast.invoke({"latitude":farm_coords["lat"],"longitude":farm_coords["lon"],"days_ahead":1})
        print(f"     precip={wx.get('precipitation_mm')}mm  rain_risk={wx.get('rain_risk')}")
        mlflow.log_metric("forecast_precip_mm", wx.get("precipitation_mm",0))
        mlflow.set_tag("rain_risk", wx.get("rain_risk","UNKNOWN"))

        print("  [4/4] RAG knowledge retrieval …")
        rag_q = "wheat irrigation " + l1["stress_level"].lower() + " soil moisture growth stage " + str(sensor_data["growth_stage"])
        rag   = agronomy_rag.invoke({"question": rag_q})
        print(f"     {rag['n_passages']} FAO-56 passages retrieved")

        # Farmer-friendly prompt (fixed: no technical units)
        soil_status  = "too dry and needs water urgently" if sensor_data["sm_root"] < SM_LOW else ("at a good level" if sensor_data["sm_root"] <= SM_HIGH else "very wet")
        pred_status  = "too dry tomorrow — crop is at risk" if l1["pred_sm_root"] < SM_LOW else ("fine tomorrow" if l1["pred_sm_root"] <= SM_HIGH else "too wet tomorrow")
        rain_note    = "IMPORTANT: Rain is coming — reduce or skip irrigation if possible." if wx.get("rain_risk") == "HIGH" else ""
        stress_note  = "The crop is at risk of stress. Act tonight." if l1["stress_level"] == "STRESSED" else ""

        prompt = f"""You are AgriSmart, an irrigation assistant for wheat farmers in Tunisia.\n\
Write 2-3 plain sentences a farmer can understand. No technical units.\n\
\n\
Today the soil is {soil_status}.\n\
Our AI predicts the soil will be {pred_status}.\n\
Rain expected tomorrow: {"yes" if wx.get("precipitation_mm",0)>1 else "no"}.\n\
\n\
Agronomy knowledge:\n\
{rag["retrieved_passages"][:400]}\n\
\n\
Decision: irrigate {str(rl["recommended_irrigation_mm"])} mm tonight.\n\
{rain_note} {stress_note}\n\
\n\
Write the recommendation now:"""

        explanation = llm.invoke(prompt).content
        mlflow.set_tag("final_action_mm", str(rl["recommended_irrigation_mm"]))
        mlflow.log_text(explanation, "explanation.txt")

        output = f"""+----------------------------------------------------------+\n\
|         AgriSmart Irrigation Recommendation              |\n\
+----------------------------------------------------------+\n\
|  Action   : Irrigate {str(rl['recommended_irrigation_mm'])} mm tonight {' '*(35-len(str(rl['recommended_irrigation_mm'])+' mm tonight'))}|\n\
|  SM today : {str(sensor_data['sm_root'])} {' '*(43-len(str(sensor_data['sm_root']))) }|\n\
|  SM tmrw  : {str(l1['pred_sm_root'])} ({l1['stress_level']}) {' '*(43-len(str(l1['pred_sm_root'])+' ('+l1['stress_level']+')'))}|\n\
|  Rain risk: {str(wx.get('rain_risk','UNKNOWN'))} {' '*(43-len(str(wx.get('rain_risk','UNKNOWN'))))}|\n\
|  Run ID   : {run.info.run_id[:8]} {' '*(43-8)}|\n\
+----------------------------------------------------------+\n\
\n\
{explanation}"""
        print(output)
        return output

print("✅ run_irrigation_agent() ready")
print("   Flow: Layer1 → RL → Weather → RAG → LLM → MLflow")

✅ run_irrigation_agent() ready
   Flow: Layer1 → RL → Weather → RAG → LLM → MLflow


## 10. Test — Stressed Soil

In [ ]:
sample_stressed = {
    "date":"2025-12-15","sm_root":0.185,"sm_shallow":0.170,"sm_deep":0.210,
    "et0_mm":3.2,"etc_mm":2.88,"kc":0.90,"precip_mm":0.0,
    "temp_c":18.5,"rh_pct":55.0,"wind_kmh":8.0,"irrigation_mm":0.0,
    "day_of_season":45,"growth_stage":1,
}
print("Stressed soil scenario:")
print("="*60)
run_irrigation_agent(sample_stressed)

Stressed soil scenario:
  [1/4] Layer 1 forecast …
     pred_sm=0.193  pred_et0=2.7008  STRESSED
  [2/4] RL advisor …
     action=10mm  constraint=None
  [3/4] Weather forecast …
     precip=0mm  rain_risk=LOW
  [4/4] RAG knowledge retrieval …


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


     3 FAO-56 passages retrieved
+----------------------------------------------------------+
|         AgriSmart Irrigation Recommendation              |
+----------------------------------------------------------+
|  Action   : Irrigate 10 mm tonight                       |
|  SM today : 0.185                                       |
|  SM tmrw  : 0.193 (STRESSED)                            |
|  Rain risk: LOW                                         |
|  Run ID   : f87246a2                                    |
+----------------------------------------------------------+

**Recommendation:**  We need to act quickly! The wheat crop is at risk of stress due to dry soil conditions. Please irrigate your fields tonight with 10mm of water.


'+----------------------------------------------------------+\n|         AgriSmart Irrigation Recommendation              |\n+----------------------------------------------------------+\n|  Action   : Irrigate 10 mm tonight                       |\n|  SM today : 0.185                                       |\n|  SM tmrw  : 0.193 (STRESSED)                            |\n|  Rain risk: LOW                                         |\n|  Run ID   : f87246a2                                    |\n+----------------------------------------------------------+\n\n**Recommendation:**  We need to act quickly! The wheat crop is at risk of stress due to dry soil conditions. Please irrigate your fields tonight with 10mm of water.'

## 11. Test — Rainy Day

In [ ]:
sample_rainy = {
    "date":"2026-01-20","sm_root":0.215,"sm_shallow":0.200,"sm_deep":0.225,
    "et0_mm":1.8,"etc_mm":1.53,"kc":0.85,"precip_mm":12.0,
    "temp_c":13.0,"rh_pct":80.0,"wind_kmh":15.0,"irrigation_mm":0.0,
    "day_of_season":81,"growth_stage":2,
}
print("Rainy day scenario:")
print("="*60)
run_irrigation_agent(sample_rainy)

Rainy day scenario:
  [1/4] Layer 1 forecast …
     pred_sm=0.2219  pred_et0=1.7455  OPTIMAL
  [2/4] RL advisor …
     action=0mm  constraint=Rain constraint: rain covers ETc demand
  [3/4] Weather forecast …
     precip=0mm  rain_risk=LOW
  [4/4] RAG knowledge retrieval …


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


     3 FAO-56 passages retrieved
+----------------------------------------------------------+
|         AgriSmart Irrigation Recommendation              |
+----------------------------------------------------------+
|  Action   : Irrigate 0 mm tonight                        |
|  SM today : 0.215                                       |
|  SM tmrw  : 0.2219 (OPTIMAL)                            |
|  Rain risk: LOW                                         |
|  Run ID   : 7715e05c                                    |
+----------------------------------------------------------+

**Recommendation:**  No need to irrigate tonight.


'+----------------------------------------------------------+\n|         AgriSmart Irrigation Recommendation              |\n+----------------------------------------------------------+\n|  Action   : Irrigate 0 mm tonight                        |\n|  SM today : 0.215                                       |\n|  SM tmrw  : 0.2219 (OPTIMAL)                            |\n|  Rain risk: LOW                                         |\n|  Run ID   : 7715e05c                                    |\n+----------------------------------------------------------+\n\n**Recommendation:**  No need to irrigate tonight.'

## 12. Conversational Chat Interface

In [ ]:
def chat_with_agent(sensor_data, farm_coords=None):
    print("Running AgriSmart analysis ...\n")
    run_irrigation_agent(sensor_data, farm_coords)

    ctx = ("Today: soil " + ("too dry" if sensor_data["sm_root"] < SM_LOW else "good") +
           ", ET0=" + str(sensor_data["et0_mm"]) + " mm/day, day " + str(sensor_data["day_of_season"]) + "/228")

    print("\n" + "="*50)
    print("Ask me anything. Type 'done' to exit.\n")

    while True:
        q = input("Farmer: ").strip()
        if q.lower() in ['done','exit','quit','']: break

        knowledge = retrieve_knowledge(q, n=2)
        p = ("You are AgriSmart, a simple irrigation assistant.\n"
             "Context: " + ctx + "\n"
             "Agronomy knowledge:\n" + knowledge + "\n\n"
             "Farmer asks: \"" + q + "\"\n"
             "Answer in 2 plain sentences:\n")
        print("AgriSmart: " + llm.invoke(p).content + "\n")

# Uncomment to run:
chat_with_agent(sample_stressed)
print("✅ chat_with_agent() ready")

Running AgriSmart analysis ...

  [1/4] Layer 1 forecast …
     pred_sm=0.193  pred_et0=2.7008  STRESSED
  [2/4] RL advisor …
     action=10mm  constraint=None
  [3/4] Weather forecast …
     precip=0mm  rain_risk=LOW
  [4/4] RAG knowledge retrieval …


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


     3 FAO-56 passages retrieved
+----------------------------------------------------------+
|         AgriSmart Irrigation Recommendation              |
+----------------------------------------------------------+
|  Action   : Irrigate 10 mm tonight                       |
|  SM today : 0.185                                       |
|  SM tmrw  : 0.193 (STRESSED)                            |
|  Rain risk: LOW                                         |
|  Run ID   : 27b4e0fa                                    |
+----------------------------------------------------------+

**Recommendation:**  We need to act quickly! The wheat crop is at risk of stress due to dry soil conditions. Please irrigate your fields tonight with 10mm of water.

Ask me anything. Type 'done' to exit.

Farmer: hello


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AgriSmart: Hello! How can I help you today?

Farmer: are we irrigating today 


...

AgriSmart: It's currently day 45 of the growing season.  To check for rain, look out your window or consult a weather forecast.

Farmer: can you check if its raining for me


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AgriSmart: I am checking the weather conditions for your location. I will let you know as soon as I have an update.

Farmer: tell me


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AgriSmart: "Your soil is currently dry and the ET0 is high at 3.2 mm/day.  Based on your location's typical monthly ET0 values, you should expect some rain soon."

Farmer: exit
✅ chat_with_agent() ready


## 13. MLflow Decision Log

In [ ]:
client = mlflow.tracking.MlflowClient()
exp    = client.get_experiment_by_name("AgriSmart_Irrigation_Agent")
if exp:
    runs = client.search_runs(exp.experiment_id, order_by=["start_time DESC"])
    print(f"Total logged decisions: {len(runs)}\n")
    print(f"{'Date':<14} {'SM':<8} {'Pred SM':<10} {'Action':>8} {'Stress':<12} {'Rain':<8}")
    print("-"*60)
    for r in runs[:10]:
        p=r.data.params; m=r.data.metrics; t=r.data.tags
        print(f"  {p.get('date','?'):<12} {p.get('sm_root','?'):<8} "
              f"{round(m.get('pred_sm_root',0),4):<10} "
              f"{int(m.get('irrigation_mm',0)):>7}mm   "
              f"{t.get('stress_level','?'):<12} {t.get('rain_risk','?')}")
else:
    print("Run the test scenarios above first.")

Total logged decisions: 3

Date           SM       Pred SM      Action Stress       Rain    
------------------------------------------------------------
  2025-12-15   0.185    0.193           10mm   STRESSED     LOW
  2026-01-20   0.215    0.2219           0mm   OPTIMAL      LOW
  2025-12-15   0.185    0.193           10mm   STRESSED     LOW


## 14. Save Configuration

In [ ]:
import json
tools_list = ["layer1_forecast","rl_advisor","weather_forecast","agronomy_rag"]
config = {
    "tools":           tools_list,
    "llm":             "google/gemma-2-2b-it",
    "rag_docs":        len(FAO56_DOCS),
    "rag_source":      "FAO-56",
    "mlflow_exp":      "AgriSmart_Irrigation_Agent",
    "sm_optimal_band": [SM_LOW, SM_HIGH],
    "actions_mm":      ACTIONS,
    "layer2":          "PyTorch Double DQN" if USE_PYTORCH else "sklearn DQN",
}
with open('agrismart_agent_config.json','w') as f:
    json.dump(config, f, indent=2)
print("✅ Saved: agrismart_agent_config.json")
print()
print("Full pipeline:")
for t in tools_list: print(f"  Tool: {t}")
print(f"  LLM : Gemma-2-2b-it (fixed: max_new_tokens=256)")
print(f"  RAG : ChromaDB + {len(FAO56_DOCS)} FAO-56 passages")
print(f"  Log : MLflow ./mlruns/")

✅ Saved: agrismart_agent_config.json

Full pipeline:
  Tool: layer1_forecast
  Tool: rl_advisor
  Tool: weather_forecast
  Tool: agronomy_rag
  LLM : Gemma-2-2b-it (fixed: max_new_tokens=256)
  RAG : ChromaDB + 13 FAO-56 passages
  Log : MLflow ./mlruns/
